In [4]:
import json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

# El cambio en este codigo es que ahora se implementa un 
# Json para que el modelo entrege un texto mas facil de copiar y pegar



client = Anthropic()
model = "claude-haiku-4-5-20251001"

# El system prompt solo define el rol
system_prompt = """Eres un tutor de matemáticas paciente y claro.
- Explica paso a paso y usa ejemplos numéricos.
- Si el estudiante se equivoca, señala el error sin desanimarlo.
- Responde en español y de forma concisa.
- Si te preguntan algo que no es de matemáticas, redirige amablemente al tema."""

# El formato JSON se pide en el mensaje del usuario
json_instruction = (
    '\n\nResponde como JSON con esta forma: '
    '{"explicacion": "...", "pasos": ["..."], "resultado": "..."}'
)

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

messages = []
history = []

while True:
    try:
        user_input = input("> ").strip()
    except (KeyboardInterrupt, EOFError):
        print("Chat terminado.")
        break

    if user_input == "" or user_input.lower() in ("salir", "exit"):
        print("Chat terminado.")
        break

    add_user_message(messages, user_input + json_instruction)
    add_assistant_message(messages, "```json")          # prefill

    with client.messages.stream(
        model=model,
        max_tokens=4000,
        system=system_prompt,
        messages=messages,
        stop_sequences=["```"],                          # corta antes del cierre
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)              # solo el interior del JSON
        final_message = stream.get_final_message()

    print()

    messages.pop()                                       # quita el prefill del historial

    response = "".join(
        block.text for block in final_message.content if block.type == "text"
    ).strip()

    if response:
        try:
            parsed = json.loads(response)
        except json.JSONDecodeError:
            parsed = None

        add_assistant_message(messages, response)
        history.append({
            "id": final_message.id,
            "model": final_message.model,
            "user_input": user_input,
            "response": response,
            "parsed": parsed,
            "stop_reason": final_message.stop_reason,
            "input_tokens": final_message.usage.input_tokens,
            "output_tokens": final_message.usage.output_tokens,
        })
    else:
        messages.pop()                                   # quita también el mensaje del usuario



{
  "explicacion": "La derivada del seno es el coseno. Esta es una de las derivadas trigonométricas más importantes que debes memorizar.",
  "pasos": [
    "Tenemos la función f(x) = sen(x)",
    "Aplicamos la regla de derivación para seno",
    "La derivada es f'(x) = cos(x)",
    "Ejemplo numérico: si f(x) = sen(x), entonces en x = 0, la derivada es cos(0) = 1"
  ],
  "resultado": "d/dx [sen(x)] = cos(x)"
}

Chat terminado.
